In [1]:
#Task 1
# Veriyi hazırla ve tabanı kur. Tiny Shakespeare'i oku, karakter tokenizer'ını
# yaz (encode, decode), veriyi %90 train, %10 val diye ayır. Rastgele
# parçalardan batch çeken get_batch fonksiyonunu yaz. Hafta 2'deki bigram
# modelini nn.Module olarak kur, eğit, val loss'u kaydet. Bu senin
# karşılaştırma tabanın. (7:52 - 38:00)

In [2]:
with open("input.txt", "r", encoding = "utf-8") as f:
    text = f.read()

In [3]:
len(text)

1115394

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [5]:
#mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

In [6]:
def encode(s):
    integers = [stoi[i] for i in s]
    return integers

def decode(l):
    lst = "".join([itos[i] for i in l])
    return lst

print(encode("hiithere"))
print(decode(encode("hiithere")))

[46, 47, 47, 58, 46, 43, 56, 43]
hiithere


In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.dtype, data.shape)
print(data[:50])

torch.int64 torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [8]:
#train val split with 90% 10%
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context=x[:t+1]
    target=y[t]
    print(f'when input is {context} the target is {target}')


when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [10]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):

    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) #do not want to grab latest datas like last data point as a starter
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [11]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)


    def forward(self, idx, targets):

        logits = self.token_embedding_table(idx) #B,T,C shape (4,8,65) rn
        B, T, C = logits.shape
        logits = logits.view(B*T,C) #now its (32,65)
        targets = targets.view(-1)
        loss = F.cross_entropy(logits,targets)
        
        return logits, loss

m = BigramLanguageModel(vocab_size)

In [12]:
#lets train our model and find our baseline
batch_size = 32
epoch = 200000

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

for _ in range(epoch):

    xb,yb = get_batch("train") #first gather batch
    logits, loss = m(xb,yb) 
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f'the train loss is {loss.item()}')

the train loss is 2.4275641441345215


In [13]:
#lets find val loss

@torch.no_grad()
def val_loss_estimator():
    losses = torch.zeros(200)
    for _ in range (200):
        x,y = get_batch("val")
        logits, loss = m(x,y)
        losses[_]= loss.item()
    print(f'the validation loss is around {losses.mean()}')
val_loss_estimator()

the validation loss is around 2.490921974182129
